## Experiment 6 — CUDA Indexing Exercises



**Objective:**  
Practice converting between multidimensional coordinates and flattened memory indices in CUDA, and strengthen understanding of how threads map to data in 1D, 2D, and image-style layouts.

The experiment will focus on:

- Computing global thread indices from `blockIdx`, `blockDim`, and `threadIdx`.
- Mapping 2D thread coordinates to matrix/image coordinates.
- Flattening `(row, col)` into a 1D row-major memory index using:
  `index = row * width + col`.
- Recovering row and column coordinates from a flattened index.
- Working with common tensor/image layouts such as HWC and CHW.
- Understanding how CUDA's `(x, y)` coordinate system maps to array coordinates `(row, col) = (y, x)`.
- Practicing bounds checks for partial blocks and non-divisible dimensions.



**Expected outcome:**  
Be able to derive indexing formulas from the data layout instead of memorizing them, and confidently map CUDA threads to elements in vectors, matrices, and images.

## Implementation

In [1]:
!nvidia-smi

Wed Sep  9 06:25:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             32W /   70W |     327MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ninja

In [3]:
# import libraries
import torch # for pytorch
from torch.utils.cpp_extension import load_inline # for loading cuda/c++ code

# versions
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")


PyTorch version: 2.11.0+cu128
CUDA available: True
cuDNN version: 91900


In [4]:
cpp_code = """
#include <torch/extension.h>

torch::Tensor index1d_launcher(torch::Tensor input);

torch::Tensor index2d_launcher(torch::Tensor input);
"""

cuda_code = """
#include <torch/extension.h>
#include <cstdint>

__global__ void index1d_kernel(const uint8_t* input, uint8_t* output, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        output[idx] = idx;
    }
}

__global__ void index2d_kernel(const uint8_t* input, uint8_t* output, int width, int height) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < height && col < width) {
        output[row * width + col] = row * width + col;
    }
}

torch::Tensor index1d_launcher(torch::Tensor input) {
    // Not implementing checks for this
    // Assuming input is a 1D tensor
    int N = input.size(0);
    torch::Tensor output = torch::zeros_like(input);
    int block_size = 256;
    int num_blocks = (N + block_size - 1) / block_size;
    index1d_kernel<<<num_blocks, block_size>>>(input.data_ptr<uint8_t>(), output.data_ptr<uint8_t>(), N);
    return output;
}


torch::Tensor index2d_launcher(torch::Tensor input) {
    // Not implementing checks for this
    // Assuming input is a 2D tensor
    int width = input.size(1);
    int height = input.size(0);
    torch::Tensor output = torch::zeros_like(input);
    dim3 block_size(16, 16);
    dim3 num_blocks((width + block_size.x - 1) / block_size.x, (height + block_size.y - 1) / block_size.y);
    index2d_kernel<<<num_blocks, block_size>>>(input.data_ptr<uint8_t>(), output.data_ptr<uint8_t>(), width, height);
    return output;
}

"""

In [5]:
module = load_inline(
    name="indexing",
    cpp_sources=cpp_code, 
    cuda_sources=cuda_code,
    functions=["index1d_launcher", "index2d_launcher"],
    extra_cuda_cflags=["-O3"],
    extra_cflags=["-O2"],
    verbose=True
)

In [8]:
A1 = torch.randint(0, 10, (10,),device='cuda',dtype=torch.uint8)
A2 = torch.randint(0, 10, (4, 4),device='cuda',dtype=torch.uint8)

In [9]:
print("A1: ", A1)
print("A2: ", A2)

A1_indexed = module.index1d_launcher(A1)
A2_indexed = module.index2d_launcher(A2)


A1:  tensor([2, 8, 7, 0, 0, 1, 5, 1, 0, 5], device='cuda:0', dtype=torch.uint8)
A2:  tensor([[6, 7, 5, 7],
        [7, 8, 2, 3],
        [7, 7, 7, 8],
        [3, 0, 9, 9]], device='cuda:0', dtype=torch.uint8)


In [10]:
print("A1_indexed: ", A1_indexed)

A1_indexed:  tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], device='cuda:0', dtype=torch.uint8)


In [11]:

print("A2_indexed: ", A2_indexed)

A2_indexed:  tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]], device='cuda:0', dtype=torch.uint8)


## Observations

- The 1D indexing kernel correctly generated consecutive global indices using:
  `idx = blockIdx.x * blockDim.x + threadIdx.x`.

- The 2D indexing kernel correctly mapped CUDA `(x, y)` coordinates to matrix `(row, col)` coordinates.

- Flattening the 2D coordinates using:
  `index = row * width + col`
  produced the expected row-major ordering.

- The experiment confirms that `x` corresponds to columns and `y` corresponds to rows in the 2D CUDA mapping used here.

- For larger index ranges, `uint8` should be avoided because values wrap after 255; a wider integer type such as `int32` is more appropriate.